# Synthetic Time Series Data Generation

Generates two tables for the MMF pipeline:

- **`train`** — historical time series with `unique_id`, `ds`, `y`, static features, and dynamic future covariates. Fed to `run_forecast(train_data=...)`.
- **`score`** — future dates with `unique_id`, `ds`, and dynamic future covariates only (no `y`, no static features). Fed to `run_forecast(scoring_data=...)`.

Covariates are generated continuously across the train/score boundary so values are consistent.

## How the data is built

**Target variable (`y`)** — base level + linear trend + seasonal sine wave + noise, then modified by signal covariates below.

**Static features** (constant per series, included in `train` only):
| Column | Values | Role |
|---|---|---|
| `static_category` | A / B / C | **Signal** — shifts base level (+50 / 0 / −30) |
| `static_region` | north / south / east / west | **Signal** — scales seasonality (north 1.5×, south 0.5×, etc.) |
| `static_color` | red / blue / green | **Noise** — no effect on `y` |

**Dynamic future numerical** (vary per series × month, included in both `train` and `score`):
| Column | How generated | Role |
|---|---|---|
| `future_planned_price` | Random walk with occasional jumps | **Signal** — higher price suppresses `y` (elasticity −0.3) |
| `future_temperature` | Seasonal sine + noise, shifted by region | **Signal** — adds to `y` proportionally |
| `future_marketing_spend` | Uniform random [0, 100] | **Noise** — no effect on `y` |

**Dynamic future categorical** (vary per series × month, included in both `train` and `score`):
| Column | Values | Role |
|---|---|---|
| `future_is_promotion` | 0 / 1 (~20% months) | **Signal** — promo months get 1.15× multiplier |
| `future_event_type` | none / minor / major | **Signal** — minor +5, major +20 to `y` |
| `future_channel` | online / instore / wholesale | **Noise** — no effect on `y` |

~10% of training series are then contaminated with spikes, drops, nulls, or gaps.
~0.1% of covariate cells in the scoring table are set to null to exercise downstream null-handling.

## 1. Parameters

In [ ]:
try:
    catalog = dbutils.widgets.get("catalog")
except Exception:
    catalog = "mmf"

try:
    schema = dbutils.widgets.get("schema")
except Exception:
    schema = "synthetic"

try:
    prediction_length = int(dbutils.widgets.get("prediction_length"))
except Exception:
    prediction_length = 12

try:
    N_SERIES = int(dbutils.widgets.get("n_series"))
except Exception:
    N_SERIES = 1000

train_table = f"{catalog}.{schema}.train"
score_table = f"{catalog}.{schema}.score"
print(f"Train table: {train_table}")
print(f"Score table: {score_table}")
print(f"Prediction length: {prediction_length} months")
print(f"Number of series: {N_SERIES:,}")

## 2. Setup — Create Catalog and Schema

In [ ]:
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {catalog}.{schema}")

## 3. Generate Synthetic Time Series

In [ ]:
import math
import pandas as pd
from datetime import date
from pyspark.sql import functions as F
from pyspark.sql.window import Window

N_YEARS = 5
N_TRAIN = N_YEARS * 12
N_TOTAL = N_TRAIN + prediction_length
TRAIN_END = date.today().replace(day=1)

train_dates = pd.date_range(end=TRAIN_END, periods=N_TRAIN, freq="MS")
score_dates = pd.date_range(start=train_dates[-1] + pd.offsets.MonthBegin(1), periods=prediction_length, freq="MS")
all_dates = list(train_dates) + list(score_dates)

dates_pdf = pd.DataFrame({
    "t": list(range(N_TOTAL)),
    "ds": [d.date() for d in all_dates],
    "is_train": [i < N_TRAIN for i in range(N_TOTAL)],
})
dates_df = spark.createDataFrame(dates_pdf)

# 1) Per-series static features and random params (one row per unique_id)
series_df = (
    spark.range(N_SERIES).withColumnRenamed("id", "series_idx")
    .withColumn("unique_id", F.format_string("series_%04d", F.col("series_idx")))
    .withColumn("static_category",
        F.element_at(
            F.array(F.lit("A"), F.lit("B"), F.lit("C")),
            (F.floor(F.rand(101) * 3) + 1).cast("int"),
        ),
    )
    .withColumn("static_region",
        F.element_at(
            F.array(F.lit("north"), F.lit("south"), F.lit("east"), F.lit("west")),
            (F.floor(F.rand(102) * 4) + 1).cast("int"),
        ),
    )
    .withColumn("static_color",
        F.element_at(
            F.array(F.lit("red"), F.lit("blue"), F.lit("green")),
            (F.floor(F.rand(103) * 3) + 1).cast("int"),
        ),
    )
    .withColumn("trend_slope", F.lit(0.5) + F.rand(104) * 4.5)
    .withColumn("level", F.lit(50.0) + F.rand(105) * 450.0)
    .withColumn("seasonality_amp", F.lit(5.0) + F.rand(106) * 45.0)
    .withColumn("noise_std", F.lit(1.0) + F.rand(107) * 19.0)
    .withColumn("price_init", F.lit(20.0) + F.rand(108) * 60.0)
)

CATEGORY_OFFSET = F.create_map(F.lit("A"), F.lit(50.0), F.lit("B"), F.lit(0.0), F.lit("C"), F.lit(-30.0))
REGION_SCALE = F.create_map(
    F.lit("north"), F.lit(1.5), F.lit("south"), F.lit(0.5),
    F.lit("east"),  F.lit(1.0), F.lit("west"),  F.lit(1.2),
)
REGION_PHASE = F.create_map(
    F.lit("north"), F.lit(0.0),         F.lit("south"), F.lit(math.pi),
    F.lit("east"),  F.lit(math.pi / 2), F.lit("west"),  F.lit(-math.pi / 2),
)
EVENT_OFFSET = F.create_map(F.lit("none"), F.lit(0.0), F.lit("minor"), F.lit(5.0), F.lit("major"), F.lit(20.0))

series_df = (
    series_df
    .withColumn("category_offset", CATEGORY_OFFSET[F.col("static_category")])
    .withColumn("region_scale",    REGION_SCALE[F.col("static_region")])
    .withColumn("region_phase",    REGION_PHASE[F.col("static_region")])
)

# 2) Cross-join series x dates -> full panel
panel = series_df.crossJoin(F.broadcast(dates_df))

# 3) Dynamic covariates
# Random walk for price: cumulative sum of N(0, 0.5) over t per series
panel = panel.withColumn("price_step", F.randn(201) * 0.5)
walk_w = Window.partitionBy("series_idx").orderBy("t").rowsBetween(Window.unboundedPreceding, Window.currentRow)
panel = (
    panel
    .withColumn("price_walk", F.sum("price_step").over(walk_w))
    .withColumn(
        "future_planned_price",
        F.greatest(F.lit(5.0), F.least(F.lit(150.0), F.col("price_walk") + F.col("price_init"))),
    )
)

# Temperature = 15 + 12*sin(2*pi*t/12 + region_phase) + N(0, 2)
panel = panel.withColumn(
    "future_temperature",
    F.lit(15.0)
    + F.lit(12.0) * F.sin(F.lit(2.0 * math.pi) * F.col("t") / F.lit(12.0) + F.col("region_phase"))
    + F.randn(202) * 2.0,
)

panel = panel.withColumn("future_marketing_spend", F.rand(203) * 100.0)
panel = panel.withColumn("future_is_promotion", (F.rand(204) < 0.20).cast("int"))

ev_r = F.rand(205)
panel = panel.withColumn(
    "future_event_type",
    F.when(ev_r < 0.05, F.lit("major")).when(ev_r < 0.20, F.lit("minor")).otherwise(F.lit("none")),
)

ch_idx = F.floor(F.rand(206) * 3).cast("int")
panel = panel.withColumn(
    "future_channel",
    F.when(ch_idx == 0, F.lit("online")).when(ch_idx == 1, F.lit("instore")).otherwise(F.lit("wholesale")),
)

# 4) y for training rows only
panel = (
    panel
    .withColumn("seasonal", F.col("region_scale") * F.col("seasonality_amp") *
                F.sin(F.lit(2.0 * math.pi) * F.col("t") / F.lit(12.0)))
    .withColumn("noise", F.randn(207) * F.col("noise_std"))
    .withColumn("event_offset", EVENT_OFFSET[F.col("future_event_type")])
    .withColumn(
        "y_raw",
        F.col("level")
        + F.col("category_offset")
        + F.col("trend_slope") * F.col("t")
        + F.col("seasonal")
        + F.lit(-0.3) * F.col("future_planned_price")
        + F.lit(0.3)  * F.col("future_temperature")
        + F.col("event_offset")
        + F.col("noise"),
    )
    .withColumn(
        "y",
        F.greatest(
            F.lit(0.0),
            F.when(F.col("future_is_promotion") == 1, F.col("y_raw") * F.lit(1.15)).otherwise(F.col("y_raw")),
        ),
    )
)

train_df = (
    panel.filter(F.col("is_train"))
    .select(
        "unique_id", "ds", "y",
        "static_category", "static_region", "static_color",
        "future_planned_price", "future_temperature", "future_marketing_spend",
        "future_is_promotion", "future_event_type", "future_channel",
        "series_idx",
    )
)

score_df = (
    panel.filter(~F.col("is_train"))
    .select(
        "unique_id", "ds",
        "future_planned_price", "future_temperature", "future_marketing_spend",
        "future_is_promotion", "future_event_type", "future_channel",
    )
)

print(f"Spark-native generation prepared (lazy): N_SERIES={N_SERIES:,}, N_TRAIN={N_TRAIN}, N_SCORE={prediction_length}")
print(f"Train date range: {all_dates[0].date()} → {all_dates[N_TRAIN - 1].date()}")
print(f"Score date range: {all_dates[N_TRAIN].date()} → {all_dates[-1].date()}")

## 3b. Inject Anomalies and Missing Entries

Contaminates ~10% of series with realistic data-quality issues:
- **Spikes** — sudden values 5–10× the local mean
- **Drops** — values forced to zero or near-zero
- **Nulls** — `y` set to `NaN` (sensor dropout / ETL failure)
- **Gaps** — entire rows removed (missing months)

In [ ]:
# Spark-native anomaly injection.
# Assign each series to one of: none / spike / drop / null / gap based on a per-series random number.
# 10% of series are affected, evenly split into the 4 anomaly types.
series_anom = (
    train_df.select("unique_id", "series_idx").distinct()
    .withColumn("anom_r", F.rand(301))
    .withColumn(
        "anomaly_type",
        F.when(F.col("anom_r") >= 0.10, F.lit("none"))
         .when(F.col("anom_r") <  0.025, F.lit("spike"))
         .when(F.col("anom_r") <  0.050, F.lit("drop"))
         .when(F.col("anom_r") <  0.075, F.lit("null"))
         .otherwise(F.lit("gap")),
    )
    .select("series_idx", "anomaly_type")
)

# Per-row randomness used to pick affected rows within a series.
# N_TRAIN = 60, so to hit ~3 spike rows we use threshold ~0.05; ~3 drop rows ~0.05;
# ~5 null rows ~0.083; ~5 gap rows ~0.083 (gap rows are dropped entirely).
SPIKE_THRESH = 0.05
DROP_THRESH  = 0.05
NULL_THRESH  = 0.083
GAP_THRESH   = 0.083

train_with_anom = (
    train_df.join(F.broadcast(series_anom), "series_idx")
    .withColumn("row_r",     F.rand(302))
    .withColumn("spike_mul", F.lit(5.0) + F.rand(303) * 5.0)
    .withColumn("drop_val",  F.rand(304))
)

train_final = (
    train_with_anom
    .withColumn(
        "y",
        F.when((F.col("anomaly_type") == "spike") & (F.col("row_r") < SPIKE_THRESH),
               F.col("y") * F.col("spike_mul"))
         .when((F.col("anomaly_type") == "drop")  & (F.col("row_r") < DROP_THRESH),
               F.col("drop_val"))
         .when((F.col("anomaly_type") == "null")  & (F.col("row_r") < NULL_THRESH),
               F.lit(None).cast("double"))
         .otherwise(F.col("y")),
    )
    .filter(~((F.col("anomaly_type") == "gap") & (F.col("row_r") < GAP_THRESH)))
    .select(
        "unique_id", "ds", "y",
        "static_category", "static_region", "static_color",
        "future_planned_price", "future_temperature", "future_marketing_spend",
        "future_is_promotion", "future_event_type", "future_channel",
    )
)

print(f"Anomaly injection wired up (lazy). Affected series: ~{int(0.10 * N_SERIES):,} / {N_SERIES:,}")
print("  spike / drop / null / gap each ~2.5% of series")
print(f"  spike rows per affected series ~{SPIKE_THRESH * N_TRAIN:.1f}")
print(f"  drop rows per affected series ~{DROP_THRESH * N_TRAIN:.1f}")
print(f"  null rows per affected series ~{NULL_THRESH * N_TRAIN:.1f}")
print(f"  gap  rows per affected series ~{GAP_THRESH * N_TRAIN:.1f} (dropped)")


## 3c. Inject Sparse Nulls into Scoring Data

Randomly sets ~1% of covariate cells to `NaN`/`None` across the scoring DataFrame.
This exercises the null-handling logic in Skill 1 (Step 6a validation).

In [ ]:
# Spark-native sparse null injection on scoring covariates (~0.1% of cells).
NULL_FRACTION = 0.001
covariate_cols = [
    "future_planned_price", "future_temperature", "future_marketing_spend",
    "future_is_promotion", "future_event_type", "future_channel",
]

score_final = score_df
for i, col in enumerate(covariate_cols):
    score_final = score_final.withColumn(
        col,
        F.when(F.rand(400 + i) < NULL_FRACTION, F.lit(None).cast(score_final.schema[col].dataType))
         .otherwise(F.col(col)),
    )

print(f"Score null injection wired up (lazy): ~{NULL_FRACTION:.1%} of each covariate column.")


## 4. Save to Delta Tables

In [ ]:
# Materialize and write to Delta. Spark plans the full DAG and executes distributed.
for name, sdf in [(train_table, train_final), (score_table, score_final)]:
    (
        sdf.write
        .format("delta")
        .mode("overwrite")
        .option("overwriteSchema", "true")
        .saveAsTable(name)
    )
    print(f"Table written: {name}")


## 5. Verify

In [ ]:
print("=== Train table ===")
display(spark.sql(f"""
    SELECT
        COUNT(*)                          AS total_rows,
        COUNT(DISTINCT unique_id)         AS n_series,
        MIN(ds)                           AS min_date,
        MAX(ds)                           AS max_date,
        ROUND(AVG(y), 2)                  AS avg_y,
        SUM(CASE WHEN y IS NULL THEN 1 ELSE 0 END) AS null_count,
        ROUND(MAX(y), 2)                  AS max_y,
        ROUND(MIN(y), 2)                  AS min_y
    FROM {train_table}
""").toPandas())

print("\nTrain covariate distributions:")
display(spark.sql(f"""
    SELECT
        COUNT(DISTINCT static_category) AS n_categories,
        COUNT(DISTINCT static_region)   AS n_regions,
        COUNT(DISTINCT static_color)    AS n_colors,
        ROUND(AVG(future_planned_price), 2)   AS avg_price,
        ROUND(AVG(future_temperature), 2)     AS avg_temp,
        ROUND(AVG(future_marketing_spend), 2) AS avg_mktg,
        ROUND(AVG(future_is_promotion), 3)    AS promo_rate,
        COUNT(DISTINCT future_event_type)     AS n_event_types,
        COUNT(DISTINCT future_channel)        AS n_channels
    FROM {train_table}
""").toPandas())

print("\n=== Score table ===")
display(spark.sql(f"""
    SELECT
        COUNT(*)                  AS total_rows,
        COUNT(DISTINCT unique_id) AS n_series,
        MIN(ds)                   AS min_date,
        MAX(ds)                   AS max_date
    FROM {score_table}
""").toPandas())

print("\nScore columns:")
spark.sql(f"DESCRIBE {score_table}").show(truncate=False)

In [ ]:
print("Train: months per series (expect 60 except gap-affected)")
spark.sql(f"""
    SELECT n_months, COUNT(*) AS n_series
    FROM (SELECT unique_id, COUNT(*) AS n_months FROM {train_table} GROUP BY unique_id)
    GROUP BY n_months ORDER BY n_months
""").show()

print(f"Score: months per series (expect {prediction_length} for all)")
spark.sql(f"""
    SELECT n_months, COUNT(*) AS n_series
    FROM (SELECT unique_id, COUNT(*) AS n_months FROM {score_table} GROUP BY unique_id)
    GROUP BY n_months ORDER BY n_months
""").show()

series_coverage = spark.sql(f"""
    SELECT
        (SELECT COUNT(DISTINCT unique_id) FROM {train_table}) AS train_series,
        (SELECT COUNT(DISTINCT unique_id) FROM {score_table}) AS score_series
""").toPandas()
print("Series coverage (should match):")
display(series_coverage)